# เทรนโมเดลคำปลุก "สายฝน" (openWakeWord v0.6.0) บน Google Colab ฟรี

## Poom ทำแค่ 4 อย่าง

1. เปิด notebook นี้ใหม่จากลิงก์ใน `WAKEWORD.md` (อย่าใช้แท็บเก่า Colab cache เซลล์เก่าไว้)
2. **Runtime → Change runtime type → T4 GPU → Save**
3. **Runtime → Run all**
4. พอเซลล์ **ขั้น 2** ขึ้นปุ่ม **Choose Files** ให้เลือก `wake-samples.zip`

จากนั้นรอจนจบ ประมาณ 1–1.5 ชั่วโมง ท้ายสุดเบราว์เซอร์จะดาวน์โหลด `saifon.onnx` ให้เอง

**ถ้าเซลล์ไหนขึ้น `NotReady` หรือ error สีแดง: หยุด แล้วส่งข้อความในกรอบนั้นมาให้ผม**
ทุกด่านตรวจถูกวางไว้ **ก่อน** งานที่นาน — ก่อนโหลดไฟล์ 17 GB และก่อนเทรน
จะได้ไม่เสียเวลาไปครึ่งชั่วโมงแล้วค่อยรู้ว่าพัง

โค้ดที่ทำงานจริงอยู่ใน [`wakeword/saifon_pipeline.py`](https://github.com/mammonrn/phone-ai-kiosk/blob/main/wakeword/saifon_pipeline.py)
ตัวเดียวกับที่ CI รันทุกครั้งที่มีการแก้ ด้วยเสียงสังเคราะห์ไม่กี่วินาที

## ขั้น 1 — ติดตั้ง และใส่ตัวแก้ openWakeWord

In [ ]:
import os, sys

# CI รัน notebook นี้ทั้งไฟล์ด้วยข้อมูลจิ๋ว ตัวแปรนี้เป็น True เฉพาะใน CI เท่านั้น
# บน Colab ไม่มีใครตั้ง SAIFON_CI จึงเป็น False เสมอ
CI = os.environ.get("SAIFON_CI") == "1"
print("Python:", sys.version.split()[0], "| CI mode" if CI else "| Colab")

# ปักรุ่นไว้ที่ v0.6.0 ไม่ใช้ main เพราะ main เปลี่ยนแล้ว
# (ไม่ clone piper-sample-generator อีกต่อไป — ใช้ stub แทน ดูด้านล่าง)
if not os.path.isdir("/content/openWakeWord"):
    !git clone -q --depth 1 --branch v0.6.0 https://github.com/dscripka/openWakeWord.git /content/openWakeWord

# โค้ด pipeline ของเราเอง (repo public ไม่ต้องใช้ token)
REPO_DIR = os.environ.get("SAIFON_REPO_DIR", "/content/phone-ai-kiosk")
if not os.path.isdir(REPO_DIR):
    !git clone -q --depth 1 https://github.com/mammonrn/phone-ai-kiosk.git {REPO_DIR}

# ทำไม --no-deps: openWakeWord v0.6.0 ประกาศ tflite-runtime เป็น dependency หลัก
# แต่ tflite-runtime มี wheel ถึงแค่ cp311 Colab เป็น 3.13 จึงลงไม่ได้ และเราไม่ใช้ tflite
!pip install -q --no-deps -e /content/openWakeWord

# dependency ที่ train.py/data.py import จริง ไล่จาก source v0.6.0
# scipy<1.17: acoustics import sph_harm ซึ่ง scipy 1.17 เอาออก
# ห้ามใส่ torch/torchaudio/torchcodec: Colab ติดคู่ที่ตรงกับ CUDA มาให้แล้ว
# export ใช้ dynamo=False (patch ใน train.py) จึงไม่ต้องใช้ onnxscript
# แต่ exporter แบบเดิมของ torch 2.11 ยังต้องมีแพ็กเกจ onnx — Colab ติดมาให้แล้ว
# ลงเองเฉพาะเมื่อไม่มี จะได้ไม่ไปแตะ protobuf ของ Colab โดยไม่จำเป็น
import importlib.util
if importlib.util.find_spec("onnx") is None:
    !pip install -q onnx
!pip install -q "scipy<1.17" onnxruntime torchinfo torchmetrics pronouncing audiomentations torch-audiomentations speechbrain mutagen acoustics datasets soundfile pyyaml

# pip -e ลง .pth ที่ Python อ่านตอนเปิด kernel เท่านั้น kernel ที่เปิดอยู่จึงยังไม่เห็น
# ใส่ path เองตรงนี้ ไม่ต้อง restart (train.py รันเป็น process ใหม่ จึงเห็นเองอยู่แล้ว)
import importlib
for p in ("/content/openWakeWord", os.path.join(REPO_DIR, "wakeword")):
    if p not in sys.path:
        sys.path.insert(0, p)
importlib.invalidate_caches()
import saifon_pipeline as sp

paths = sp.Paths()
print("stub:", sp.write_piper_stub(paths))
for line in sp.patch_train_py(paths):
    print("train.py ->", line)
print("torch_audiomentations ->", sp.patch_torch_audiomentations())

### ขั้น 1ก — ตรวจความพร้อมก่อนทำอะไรที่นาน

ไม่ผ่านจะหยุดทันที (tflite ไม่ต้องมี — ตรวจว่าไม่มีใครต้องใช้มัน)

In [ ]:
import importlib, importlib.util, inspect, platform
import numpy as np

problems = []

# import openwakeword.train ลาก openwakeword.data มาด้วย ซึ่ง import speechbrain,
# audiomentations, pronouncing, mutagen, acoustics ที่ระดับไฟล์
for name in ("openwakeword", "openwakeword.utils", "openwakeword.data",
             "openwakeword.train"):
    try:
        importlib.import_module(name)
        print(f"  ok   {name}")
    except Exception as exc:
        problems.append(f"{name}: {type(exc).__name__}: {exc}")
        print(f"  FAIL {name}  <- {type(exc).__name__}: {exc}")

for name in ("torch", "torchaudio", "torchcodec", "torchinfo", "torchmetrics",
             "onnxruntime", "numpy", "scipy", "sklearn", "yaml", "tqdm",
             "speechbrain", "audiomentations", "torch_audiomentations",
             "pronouncing", "mutagen", "acoustics", "datasets", "soundfile", "onnx"):
    try:
        importlib.import_module(name)
    except Exception as exc:
        problems.append(f"{name}: {type(exc).__name__}: {exc}")
        print(f"  FAIL {name}  <- {type(exc).__name__}: {exc}")

import scipy
if tuple(int(x) for x in scipy.__version__.split(".")[:2]) >= (1, 17):
    problems.append(f"scipy {scipy.__version__} ไม่มี sph_harm แล้ว acoustics จะพัง")

from openwakeword.utils import AudioFeatures
framework = inspect.signature(AudioFeatures.__init__).parameters["inference_framework"].default
if framework != "onnx":
    problems.append(f"AudioFeatures default เป็น {framework} ไม่ใช่ onnx")
print("tflite_runtime ติดตั้งอยู่:", importlib.util.find_spec("tflite_runtime") is not None, "(ควรเป็น False)")

import torch, torchaudio
print("torch", torch.__version__, "| torchaudio", torchaudio.__version__,
      "| torchaudio.info มีไหม:", hasattr(torchaudio, "info"))
if not CI:
    # ตรวจ "หลัง" pip เพราะ pip อาจสลับ torch เป็น build CPU แล้ว T4 หายเงียบๆ
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        problems.append("มองไม่เห็น GPU — Runtime > Change runtime type > T4 GPU "
                        "แล้ว Runtime > Disconnect and delete runtime แล้ว Run all ใหม่")
    # ACAV 17.28 GB + feature ของเรา + เผื่อ
    if sp.free_gb(paths.root) < 25:
        problems.append(f"ดิสก์ว่าง {sp.free_gb(paths.root):.1f} GB ต้องอย่างน้อย 25 GB")

# โมเดลดึง feature: train.py ไม่ได้เรียก download_models() ให้ และ repo ไม่ได้แถมมา
import openwakeword, openwakeword.utils, pathlib
openwakeword.utils.download_models(model_names=["melspectrogram"])
models_dir = pathlib.Path(openwakeword.__file__).parent / "resources" / "models"
for need in ("melspectrogram.onnx", "embedding_model.onnx"):
    if not (models_dir / need).exists():
        problems.append(f"ขาด {need}")

sp.fail(problems, "แก้ก่อน อย่ารันต่อ")

# ด่านที่รอบก่อนพังช้า — ตรวจตอนนี้เลย ไม่ต้องรอ 40 นาที:
sp.check_frame_math()      # total_length 32000 -> 16 frames ทั้งฝั่ง feature และฝั่งโมเดล
probe = paths.root / "probe.wav"
sp.write_wav(probe, (np.random.randn(16000) * 1000).astype(np.int16))
sp.check_torch_audiomentations_reads(probe)   # เส้นทาง torchaudio.info ที่พังบน Colab
sp.check_onnx_export()     # export แบบเดียวกับท้าย train.py บนโมเดลจิ๋ว ก่อนเทรนจริง 1 ชม.
print("\n  พร้อม ไปขั้น 2 ได้")

## ขั้น 2 — อัปโหลด wake-samples.zip

กด **Choose Files** แล้วเลือก `wake-samples.zip` (ไฟล์เดิมจาก VPS ใช้ได้ ไม่ต้องสร้างใหม่)

เซลล์นี้คัด positive **เฉพาะข้อความที่จบด้วย "สายฝน"** (อ่านข้อความจาก `manifest.csv`)
ตัดความเงียบหัวท้าย และแบ่งชุดทดสอบตามเสียงผู้พูด

In [ ]:
import zipfile, shutil
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise sp.NotReady("ไม่ได้เลือกไฟล์")
zip_name = next(iter(uploaded))
shutil.rmtree(paths.raw, ignore_errors=True)
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(paths.raw)

# รอบคูณไฟล์: train.py สร้าง feature เท่าจำนวน "ไฟล์" ไม่ใช่ ไฟล์ x augmentation_rounds
# จึงคัดลอกไฟล์จริง แต่ละสำเนาได้ augmentation สุ่มคนละแบบ
POSITIVE_ROUNDS = 2 if CI else 25
NEGATIVE_ROUNDS = 1 if CI else 10
AUGMENT_BATCH = 4 if CI else 16

report = sp.prepare_clips(paths, positive_rounds=POSITIVE_ROUNDS, negative_rounds=NEGATIVE_ROUNDS)
print("positive ที่ใช้:")
for text, n in report.kept_texts.items():
    print(f"   {n:4}  {text}")
print("positive ที่ตัดทิ้ง (มีคำต่อท้ายคำปลุก):")
for text, n in report.dropped_texts.items():
    print(f"   {n:4}  {text}")
print("ยาวเกินหน้าต่าง 2 วินาที ตัดทิ้ง:", report.too_long)
print("positive ยาวสุดหลังตัดความเงียบ:", report.longest_positive, "samples")
print("เสียงที่กันไว้ทดสอบ:", report.held_out_voices)
for k in sp.SPLITS:
    print(f"  {k:<15} {report.originals[k]:5} คลิป  -> {report.counts[k]:6} ไฟล์หลังคูณรอบ")

sp.check_split(report, min_positive_train=3 if CI else 150, min_each=1 if CI else 20,
               augmentation_batch_size=AUGMENT_BATCH)
print("\n  ข้อมูลเสียงของเราผ่าน")

### ขั้น 3ก — ตัวอ่านเสียงที่รองรับทั้งรูปแบบเก่าและใหม่

`datasets` รุ่นใหม่คืน `torchcodec.decoders.AudioDecoder` แทน dict เซลล์นี้มีแต่ฟังก์ชัน รันแล้วไม่เกิดอะไรขึ้น

In [ ]:
# ทั้งเซลล์นี้มีแต่ฟังก์ชัน ไม่มี side effect — CI ดึงเซลล์นี้ไปรันเทสต์ตรงๆ
import numpy as np
from math import gcd

TARGET_SR = 16000


def audio_16k_mono(value, target_sr=TARGET_SR):
    """คืนเสียงเป็น float32 mono ที่ target_sr ไม่ว่าข้อมูลจะมาในรูปแบบไหน

    ทำไมต้องมีฟังก์ชันนี้: `datasets` 5.x คืน torchcodec AudioDecoder แทน dict
    ตัว wrapper ของ datasets ยังรองรับ `["array"]` กับ `["sampling_rate"]` อยู่
    แต่คีย์อื่นจะโยน TypeError ทิ้ง ซึ่งเป็นสาเหตุที่ `row["audio"]["path"]`
    ตายด้วย "object is not subscriptable" ทั้งที่ส่วนอื่นของ notebook ปกติดี

    แทนที่จะเลือกข้างว่าจะพึ่งรูปแบบเก่าหรือใหม่ ฟังก์ชันนี้ดูว่าได้อะไรมาจริง
    """
    samples = None
    source_sr = None

    # 1. torchcodec AudioDecoder (datasets >= 4) — เรียก API ตัวจริง ไม่ใช่ shim
    #    เพราะมันบอก sample rate จริงและ channel จริง ไม่ใช่ค่าที่เฉลี่ยมาแล้ว
    if hasattr(value, "get_all_samples"):
        block = value.get_all_samples()
        data = block.data.cpu().numpy()          # (channels, samples) float ใน [-1, 1]
        samples = data.mean(axis=0) if data.ndim > 1 else data
        source_sr = int(block.sample_rate)

    # 2. dict — ทั้งแบบ decode แล้ว (เก่า) และแบบยังไม่ decode
    elif isinstance(value, dict):
        if value.get("array") is not None:
            samples = np.asarray(value["array"], dtype=np.float32)
            if samples.ndim > 1:
                samples = samples.mean(axis=0 if samples.shape[0] < samples.shape[-1] else -1)
            source_sr = int(value.get("sampling_rate") or target_sr)
        else:
            samples, source_sr = _decode(value.get("bytes") or value.get("path"), target_sr)

    # 3. path หรือ bytes ดิบ
    elif isinstance(value, (str, bytes, bytearray)):
        samples, source_sr = _decode(value, target_sr)

    # 4. อะไรที่ subscript ได้แต่ไม่ใช่ dict — คือ shim ของ datasets เผื่อวันหนึ่ง
    #    เมธอด get_all_samples หายไปจากใต้เท้าเรา
    else:
        try:
            samples = np.asarray(value["array"], dtype=np.float32)
            source_sr = int(value["sampling_rate"])
        except Exception as exc:
            raise TypeError(f"ไม่รู้จักรูปแบบเสียงนี้: {type(value).__name__}") from exc

    samples = np.asarray(samples, dtype=np.float32).reshape(-1)
    if samples.size == 0:
        raise ValueError("ไม่มีตัวอย่างเสียงในรายการนี้")
    return _resample(samples, source_sr, target_sr), target_sr


def _decode(source, target_sr):
    """bytes หรือ path ที่ยังไม่ decode -> samples ขอ 16 kHz mono ตั้งแต่ตอน decode"""
    if source is None:
        raise ValueError("รายการนี้ไม่มีทั้ง bytes และ path")
    from torchcodec.decoders import AudioDecoder

    # ใส่ sample_rate/num_channels ตรงนี้ = ให้ torchcodec resample ตอน decode
    # ซึ่งดีกว่า decode ที่ rate เดิมแล้วมา resample ทีหลัง
    block = AudioDecoder(source, sample_rate=target_sr, num_channels=1).get_all_samples()
    data = block.data.cpu().numpy()
    return (data.mean(axis=0) if data.ndim > 1 else data), int(block.sample_rate)


def _resample(samples, source_sr, target_sr):
    """resample ด้วย scipy ไม่เพิ่ม dependency เพราะ scipy มีอยู่แล้ว

    เซลล์เดิมเขียนไฟล์ด้วย header 16000 แบบฮาร์ดโค้ดโดยไม่เคยดูว่าเสียงต้นทาง
    เป็น rate เท่าไร ถ้า dataset ไม่ได้เป็น 16 kHz อยู่แล้วพอดี impulse response
    ทุกไฟล์จะเพี้ยนระดับเสียงแบบเงียบๆ โดยไม่มีใครรู้
    """
    if not source_sr or source_sr == target_sr:
        return samples
    from scipy.signal import resample_poly

    divisor = gcd(int(source_sr), int(target_sr))
    return resample_poly(samples, target_sr // divisor, source_sr // divisor).astype(np.float32)


def to_int16(samples):
    """float [-1, 1] -> int16 โดย clip ก่อน

    ต้อง clip: resample_poly แกว่งเกิน 1.0 ได้ที่ขอบของสัญญาณ และ
    `(x * 32767).astype(np.int16)` จะ wrap รอบตัวเลขกลายเป็นเสียงแตกดังลั่น
    ซึ่งเป็นความเสียหายที่มองไม่เห็นจากจำนวนไฟล์ ต้องฟังเท่านั้นจึงจะรู้
    """
    return (np.clip(samples, -1.0, 1.0) * 32767).astype(np.int16)


print("ตัวอ่านเสียงพร้อม: audio_16k_mono(), to_int16()")

## ขั้น 4 — เสียงก้องห้องและเสียงพื้นหลัง (ประมาณ 10–20 นาที)

ความหลากหลายของเสียงรบกวน เสียงก้อง และระดับเสียงถูกสร้างขึ้นตรงนี้ฟรี
คำเตือน "unauthenticated" ของ Hugging Face ไม่ต้องสนใจ ทั้งสองชุดเป็น public ไม่ต้องใส่ token

In [ ]:
import numpy as np
import scipy.io.wavfile

for directory in (paths.rirs, paths.background):
    shutil.rmtree(directory, ignore_errors=True)
    directory.mkdir(parents=True, exist_ok=True)

if CI:
    # CI ห้ามโหลด dataset: สร้าง impulse response กับเสียงพื้นหลังปลอมไม่กี่ไฟล์
    rng = np.random.default_rng(0)
    for i in range(3):
        ir = np.zeros(4000, dtype=np.float32); ir[0] = 1.0
        ir[1:] += rng.normal(0, 0.05, 3999) * np.exp(-np.arange(3999) / 600)
        scipy.io.wavfile.write(str(paths.rirs / f"rir_{i:05d}.wav"), 16000, to_int16(ir * 0.9))
    for i in range(4):
        scipy.io.wavfile.write(str(paths.background / f"bg_{i:05d}.wav"), 16000,
                               to_int16(rng.normal(0, 0.1, 16000 * 3).astype(np.float32)))
else:
    import datasets
    rir = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses",
                                split="train", streaming=True)
    written = skipped = 0
    for i, row in enumerate(rir):
        try:
            wave_, sr = audio_16k_mono(row["audio"])
        except Exception as exc:
            skipped += 1
            if skipped <= 3:
                print(f"  ข้าม RIR #{i}: {type(exc).__name__}: {exc}")
            continue
        scipy.io.wavfile.write(str(paths.rirs / f"rir_{i:05d}.wav"), sr, to_int16(wave_))
        written += 1
    print(f"RIR: เขียน {written} ข้าม {skipped}")

    audioset = datasets.load_dataset("agkphysics/AudioSet", split="train", streaming=True)
    written = skipped = too_short = 0
    for i, row in enumerate(audioset):
        if written >= 800:
            break
        try:
            wave_, sr = audio_16k_mono(row["audio"])
        except Exception as exc:
            skipped += 1
            if skipped <= 3:
                print(f"  ข้ามคลิป #{i}: {type(exc).__name__}: {exc}")
            continue
        if len(wave_) < sr:
            too_short += 1
            continue
        scipy.io.wavfile.write(str(paths.background / f"bg_{written:05d}.wav"), sr,
                               to_int16(wave_[: sr * 10]))
        written += 1
    print(f"background: เขียน {written} ข้าม {skipped} สั้นเกินไป {too_short}")

### ขั้น 4ก — ตรวจเสียงรบกวน และโหลดชุดวัดปลุกผิด (ไฟล์เล็ก)

In [ ]:
VALIDATION_URL = ("https://huggingface.co/datasets/davidscripka/openwakeword_features/"
                  "resolve/main/validation_set_features.npy")
if CI:
    np.save(paths.validation, np.random.default_rng(1).normal(0, 1, (3000, 96)).astype(np.float32))
elif not paths.validation.exists() or paths.validation.stat().st_size < 100_000:
    !wget -q -O {paths.validation} {VALIDATION_URL}

problems = []
for label, directory, need in (("RIR", paths.rirs, 3 if CI else 100),
                               ("เสียงพื้นหลัง", paths.background, 4 if CI else 200)):
    count = len(list(directory.glob("*.wav")))
    print(f"  {label:<14} {count} ไฟล์")
    if count < need:
        problems.append(f"{label}: {count} ไฟล์ ต้องอย่างน้อย {need} — รันขั้น 4 ใหม่")
sp.fail(problems, "อย่าไปต่อ")
# torch_audiomentations จะอ่าน header ของไฟล์พื้นหลังทุกไฟล์ด้วยเส้นทางที่ patch ไว้
sp.check_torch_audiomentations_reads(next(paths.background.glob("*.wav")))
print("  validation features:", sp.check_validation(paths, min_rows=1000 if CI else 100_000))
print("\n  ครบ ไปขั้น 5 ได้")

## ขั้น 5 — config แล้ว augment + สร้าง feature (ก่อนโหลด ACAV)

`--augment_clips` ไม่ใช้ไฟล์ ACAV เลย จึงทำก่อนได้ และตรวจ shape ของ feature ทุกชุด
**ก่อน**โหลดไฟล์ 17 GB — ถ้าผิดจะรู้ตอนนี้ ไม่ใช่หลังรอโหลดครึ่งชั่วโมง

ลบ feature เก่าทิ้งทุกครั้ง (ลบเฉพาะ 4 ไฟล์ตามชื่อ ในโฟลเดอร์ `saifon` เท่านั้น)
เพราะ feature ค้างครึ่งทางทำให้ train.py ข้าม augmentation แล้วหาไฟล์ไม่เจอ

In [ ]:
config_text = sp.write_config(
    paths,
    steps=60 if CI else 20000,
    augmentation_batch_size=AUGMENT_BATCH,
    batch_n_per_class=({"ACAV100M_sample": 64, "adversarial_negative": 16, "positive": 16} if CI
                       else {"ACAV100M_sample": 1024, "adversarial_negative": 50, "positive": 50}))
print(config_text)

print("ลบ feature เก่า:", sp.clean_features(paths) or "ไม่มี")
sp.run_train(paths, "augment")
for name, shape in sp.check_features(paths, report).items():
    print(f"  ok   {name:<30} {shape}")
print("\n  feature ทุกชุดมี", sp.FRAMES, "frames ตรงกับ ACAV")

## ขั้น 6 — โหลด ACAV100M (≈17.28 GB, 10–30 นาที)

ถ้าหลุดกลางทาง รันเซลล์นี้ซ้ำได้ `wget -c` โหลดต่อจากที่ค้าง

In [ ]:
ACAV_URL = ("https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/"
            "openwakeword_features_ACAV100M_2000_hrs_16bit.npy")
if CI:
    fake = np.lib.format.open_memmap(paths.acav, mode="w+", dtype=np.float16, shape=(4000, 16, 96))
    fake[:] = np.random.default_rng(2).normal(0, 1, fake.shape).astype(np.float16)
    fake.flush(); del fake
    print("ACAV:", sp.check_acav(paths, min_bytes=0))
else:
    have = paths.acav.stat().st_size if paths.acav.exists() else 0
    need_gb = (sp.ACAV_MIN_BYTES + 400_000_000 - have) / 1e9
    if sp.free_gb(paths.root) < need_gb + 2:
        raise sp.NotReady(f"ดิสก์ว่าง {sp.free_gb(paths.root):.1f} GB ไม่พอโหลดอีก {need_gb:.1f} GB")
    !wget -c -q --show-progress -O {paths.acav} {ACAV_URL}
    print("ACAV:", sp.check_acav(paths), f"{paths.acav.stat().st_size / 1e9:.2f} GB")

## ขั้น 7 — เทรน (30–60 นาที ปล่อยแท็บเปิดไว้)

In [ ]:
sp.run_train(paths, "train")
model_path = sp.find_model(paths)
print("\nโมเดล:", model_path, f"{model_path.stat().st_size / 1024:.1f} KB")

## ขั้น 8 — วัดผลเบื้องต้น (ทีละ 1 หน้าต่าง แบบเดียวกับมือถือ)

โมเดลรับ input `[1, 16, 96]` เท่านั้น จึงวัดทีละรายการ

In [ ]:
evaluation = sp.evaluate(paths, validation_limit=None)
print("input shape:", evaluation.input_shape)
print(f"positive_test {len(evaluation.pos_scores)} | negative_test {len(evaluation.neg_scores)}"
      f" | ชุดวัดปลุกผิด {evaluation.val_hours:.1f} ชม.\n")
print(evaluation.table())
print("\nเกณฑ์: recall ≥ 90% และปลุกผิด ≤ 0.125 ครั้ง/ชม. — ตัวเลขนี้ยังไม่ใช่ผลบน A07")

> ## 🔴 ตัวเลขข้างบนยังไม่ใช่การผ่านเกณฑ์
>
> เกณฑ์คือ **แม่น ≥90% ที่ 3 เมตร** และ **ปลุกผิดไม่เกิน 1 ครั้งต่อ 8 ชั่วโมง**
> วัดได้บน Galaxy A07 เครื่องจริงเท่านั้น ตัวเลขใน Colab มาจากเสียงสังเคราะห์
> ใช้บอกแค่ว่าคุ้มจะเอาลงเครื่องไปวัดต่อหรือไม่

## ขั้น 9 — ดาวน์โหลด saifon.onnx

In [ ]:
from google.colab import files
model_path = sp.find_model(paths)
print("ขนาด:", round(model_path.stat().st_size / 1024, 1), "KB")
files.download(str(model_path))
print("ส่งตารางจากขั้น 8 มาให้ผม พร้อมบอกว่าได้ไฟล์ saifon.onnx แล้ว")